In [102]:
from pathlib import Path
import torch
import torch.nn as nn 
from torch.utils.data import Dataset, DataLoader
import re

In [103]:
def load_data(folder_path):
    files_list = []
    # Loop through all items in the folder
    for file_path in folder_path.iterdir():
        # Check if the item is a file (skips subfolders)
        if file_path.is_file():
            if(file_path.suffix == ".c"):
                # Open and read the file content
                content = file_path.read_text(encoding="utf-8")
                label = "c"
                files_list.append((content, label))
            elif(file_path.suffix == ".java"):
                # Open and read the file content
                content = file_path.read_text(encoding="utf-8")
                label = "java"
                files_list.append((content, label))

    return files_list


In [ ]:
folder_path = Path("training_data")

data = load_data(folder_path)

In [105]:

output = []
for item in data:
    output += (re.split(r',|;|\n| ',item[0]))

dictionary = {}
i = 0
for word in output:
    if word =="" or word in dictionary:
        continue
    dictionary[word] = i
    i+=1


In [106]:
def text_to_bow(text, dictionary):
    Bag_of_Words = [0] * len(dictionary)

    output = re.split(r'([{}()\[\];,=+\-*/<>!])|\s+',text)

    for word in output:
        if word in dictionary:
            Bag_of_Words[dictionary[word]] +=1

    Bag_of_Words_tensor = torch.tensor(Bag_of_Words, dtype=torch.float32)
    
    return Bag_of_Words_tensor

In [107]:
class CodeDataset(Dataset):
    def __init__(self, data_list):
        self.data_list = data_list
        
    def __len__(self):
        return len(self.data_list)
        
    def __getitem__(self, idx):
        Bag_of_Words_tensor = text_to_bow(self.data_list[idx][0], dictionary)
        label = 1 if (self.data_list[idx][1] == 'c') else 0

        return Bag_of_Words_tensor, torch.tensor(label, dtype=torch.long)

In [108]:
dataset = CodeDataset(data)
dataload =  DataLoader(dataset, batch_size=4, shuffle=True)

# نأخذ أول "باقة" (Batch) من الـ DataLoader
first_batch = next(iter(dataload))

#print("Shape of inputs:", first_batch[0].shape)
#print("Shape of labels:", first_batch[1].shape)


In [109]:
class CodeClassifier(nn.Module):
    def __init__(self,input_size):
        super().__init__()
        hidden_size = 128
        num_classes = 2
        self.l1 = nn.Linear(input_size, hidden_size)
        self.activation = nn.ReLU()
        self.l2 = nn.Linear(hidden_size, num_classes)
    
    def forward(self,x):
        output = self.l1(x)
        output = self.activation(output)
        output = self.l2(output)

        return output


In [110]:
vocab_size = len(dictionary)
model = CodeClassifier(input_size=vocab_size)
criterion = nn.CrossEntropyLoss() 
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
epochs = 20

In [111]:
for epoch in range(epochs):
    for inputs, labels in dataload:

        outputs = model(inputs)
        loss = criterion(outputs, labels)
        

        optimizer.zero_grad()
        loss.backward() 
        optimizer.step()
        
    print(f"Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.4f}")
    

Epoch [1/20], Loss: 1.0073
Epoch [2/20], Loss: 0.0730
Epoch [3/20], Loss: 0.0058
Epoch [4/20], Loss: 0.0006
Epoch [5/20], Loss: 0.0001
Epoch [6/20], Loss: 0.0000
Epoch [7/20], Loss: 0.0000
Epoch [8/20], Loss: 0.0000
Epoch [9/20], Loss: 0.0000
Epoch [10/20], Loss: 0.0000
Epoch [11/20], Loss: 0.0000
Epoch [12/20], Loss: 0.0000
Epoch [13/20], Loss: 0.0000
Epoch [14/20], Loss: 0.0000
Epoch [15/20], Loss: 0.0000
Epoch [16/20], Loss: 0.0000
Epoch [17/20], Loss: 0.0000
Epoch [18/20], Loss: 0.0000
Epoch [19/20], Loss: 0.0000
Epoch [20/20], Loss: 0.0000


In [112]:

test_code_c = "#include <stdio.h> int main() { return 0; }"
test_code_j ="public Book(String id, String title, String author, String category, int year, int copies) { this.ID = id; this.Title = title; this.Author = author; this.Category = category; this.Year = year; this.Copies = copies;"

Bag_of_Words_tensor = text_to_bow(test_code_j,dictionary)
Bag_of_Words_tensor = Bag_of_Words_tensor.unsqueeze(0)

pred = model(Bag_of_Words_tensor)
res = torch.argmax(pred, dim=1)
print("c") if (res == 1) else print("java")

c
